In [1]:
# Cell 1 — Mount Drive and install packages
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/PhishGuard'

import os, subprocess
subprocess.run(['pip', 'install',
    'xgboost==2.1.0', 'shap==0.45.0', 'scikit-learn==1.5.0',
    'pandas==2.2.0', 'numpy==1.26.0', 'joblib==1.4.0', '-q'], check=False)

assert os.path.exists(f'{DRIVE_BASE}/features/contract_features.csv'), \
    'contract_features.csv not found — run Notebook 03 first'
assert os.path.exists(f'{DRIVE_BASE}/models/contract_feature_schema.json'), \
    'contract_feature_schema.json not found — run Notebook 03 first'
print('Cell 1 ready.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cell 1 ready.


In [4]:
# Cell 2 — Load features and schema, validate inputs
import pandas as pd
import numpy as np
import json
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
import xgboost as xgb
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

MODEL_PATH  = f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl'
SCHEMA_PATH = f'{DRIVE_BASE}/models/contract_feature_schema.json'
EVAL_DIR    = f'{DRIVE_BASE}/evaluation'

with open(SCHEMA_PATH) as f:
    FEATURE_COLS = json.load(f)

df = pd.read_csv(f'{DRIVE_BASE}/features/contract_features.csv')

assert df.shape == (1198, 23),                f'Wrong shape: {df.shape}'
assert df['label'].value_counts()[1] == 599,  'Phishing count wrong'
assert df['label'].value_counts()[0] == 599,  'Benign count wrong'
assert df.isnull().sum().sum() == 0,          'Nulls found'
assert len(FEATURE_COLS) == 21,               f'Schema length wrong: {len(FEATURE_COLS)}'
assert all(c in df.columns for c in FEATURE_COLS), 'Missing feature columns'

# ── Bytecode leakage check ────────────────────────────────────────────────────
# All phishing contracts in this dataset are self-destructed (bytecode_size==0).
# All benign contracts have bytecode. Bytecode-derived features are therefore
# perfect separators and must be excluded from training.
bytecode_zero_by_label = df.groupby('label')['bytecode_size'].apply(lambda x: (x == 0).mean())
print('Fraction with bytecode_size==0 per label (0=benign, 1=phishing):')
print(bytecode_zero_by_label.to_string())
assert bytecode_zero_by_label[1] == 1.0, \
    'Unexpected: not all phishing contracts are self-destructed — re-check leakage'
print()

# ── Drop all bytecode-derived features ───────────────────────────────────────
BYTECODE_FEATURES = [
    'bytecode_size',
    'opcode_freq_CALL', 'opcode_freq_DELEGATECALL', 'opcode_freq_SELFDESTRUCT',
    'opcode_freq_SSTORE', 'opcode_freq_JUMPI', 'external_call_sites_count',
    'has_create2', 'proxy_pattern_detected',
    'approval_then_external_call_pattern', 'approval_then_state_mutation_pattern',
    'control_flow_complexity_score'
]
TRAIN_FEATURES = [c for c in FEATURE_COLS if c not in BYTECODE_FEATURES]

print(f'Total schema features:   {len(FEATURE_COLS)}')
print(f'Bytecode features dropped: {len(BYTECODE_FEATURES)}')
print(f'Training features kept:  {len(TRAIN_FEATURES)}')
print('Training features:')
for f in TRAIN_FEATURES:
    print(f'  {f}')

print(f'\nLoaded: {df.shape}')
print(f'Label distribution:\n{df["label"].value_counts().to_string()}')
print('Cell 2 validation passed.')


Fraction with bytecode_size==0 per label (0=benign, 1=phishing):
label
0    0.0
1    1.0

Total schema features:   21
Bytecode features dropped: 12
Training features kept:  9
Training features:
  is_verified
  abi_function_count
  external_public_function_count
  approval_related_function_flag
  permit_related_function_flag
  setApprovalForAll_flag
  slither_warning_count_total
  slither_low_level_call_count
  slither_access_control_issues_count

Loaded: (1198, 23)
Label distribution:
label
0    599
1    599
Cell 2 validation passed.


In [5]:
# Cell 3 — Train/test split
X = df[TRAIN_FEATURES].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape}  |  phishing={y_train.sum()}  benign={(y_train==0).sum()}')
print(f'Test:  {X_test.shape}   |  phishing={y_test.sum()}   benign={(y_test==0).sum()}')
print('Cell 3 split complete.')



Train: (958, 9)  |  phishing=479  benign=479
Test:  (240, 9)   |  phishing=120   benign=120
Cell 3 split complete.


In [6]:
# Cell 4 — XGBoost training with 5-fold cross-validation
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.7,
    min_child_weight=1,
    gamma=0,
    reg_alpha=0,
    reg_lambda=0.5,
    scale_pos_weight=1,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    model, X_train, y_train,
    cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score=False,
    n_jobs=-1
)

print('=== 5-Fold Cross-Validation (train set) ===')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    scores = cv_results[f'test_{metric}']
    print(f'  {metric:<12}  mean={scores.mean():.4f}  std={scores.std():.4f}')

# Sanity check: CV scores of 1.0000 still indicate leakage
for metric in ['accuracy', 'f1', 'roc_auc']:
    mean_score = cv_results[f'test_{metric}'].mean()
    assert mean_score < 0.9999, \
        f'CV {metric}={mean_score:.4f} — still detecting leakage. ' \
        f'Check TRAIN_FEATURES for remaining bytecode-correlated columns.'

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
print('\nFinal model trained on full train set.')
print('Cell 4 training complete.')


=== 5-Fold Cross-Validation (train set) ===
  accuracy      mean=0.8580  std=0.0152
  precision     mean=0.8778  std=0.0245
  recall        mean=0.8330  std=0.0257
  f1            mean=0.8543  std=0.0160
  roc_auc       mean=0.9127  std=0.0184

Final model trained on full train set.
Cell 4 training complete.


In [7]:
# Cell 5 — Evaluation on held-out test set
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)
cm   = confusion_matrix(y_test, y_pred)

print('=== TEST SET EVALUATION ===')
print(f'  Accuracy:  {acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall:    {rec:.4f}')
print(f'  F1:        {f1:.4f}')
print(f'  ROC-AUC:   {auc:.4f}')
print(f'\nConfusion Matrix:\n{cm}')
print(f'\n{classification_report(y_test, y_pred, target_names=["benign","phishing"])}')

assert f1  >= 0.70, f'F1 too low: {f1:.4f}'
assert auc >= 0.75, f'AUC too low: {auc:.4f}'
print('Minimum performance thresholds passed.')



=== TEST SET EVALUATION ===
  Accuracy:  0.8792
  Precision: 0.8824
  Recall:    0.8750
  F1:        0.8787
  ROC-AUC:   0.9156

Confusion Matrix:
[[106  14]
 [ 15 105]]

              precision    recall  f1-score   support

      benign       0.88      0.88      0.88       120
    phishing       0.88      0.88      0.88       120

    accuracy                           0.88       240
   macro avg       0.88      0.88      0.88       240
weighted avg       0.88      0.88      0.88       240

Minimum performance thresholds passed.


In [8]:
# Cell 6 — SHAP feature importance
os.makedirs(EVAL_DIR, exist_ok=True)

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary bar plot — mean |SHAP| per feature
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test,
                  feature_names=TRAIN_FEATURES,
                  plot_type='bar',
                  show=False)
plt.tight_layout()
bar_path = f'{EVAL_DIR}/contract_shap_bar.png'
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {bar_path}')

# Beeswarm plot
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test,
                  feature_names=TRAIN_FEATURES,
                  show=False)
plt.tight_layout()
bee_path = f'{EVAL_DIR}/contract_shap_beeswarm.png'
plt.savefig(bee_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {bee_path}')

# Top features by mean |SHAP|
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx       = np.argsort(mean_abs_shap)[::-1]
print('\nFeature importance (mean |SHAP|):')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank}. {TRAIN_FEATURES[idx]:<45}  {mean_abs_shap[idx]:.5f}')

print('Cell 6 SHAP complete.')


Saved: /content/drive/MyDrive/PhishGuard/evaluation/contract_shap_bar.png
Saved: /content/drive/MyDrive/PhishGuard/evaluation/contract_shap_beeswarm.png

Feature importance (mean |SHAP|):
  1. is_verified                                    0.94075
  2. slither_warning_count_total                    0.77809
  3. abi_function_count                             0.56732
  4. approval_related_function_flag                 0.55025
  5. external_public_function_count                 0.40674
  6. slither_low_level_call_count                   0.15485
  7. setApprovalForAll_flag                         0.06543
  8. permit_related_function_flag                   0.01251
  9. slither_access_control_issues_count            0.00000
Cell 6 SHAP complete.


In [11]:
# Cell 7 — Calibrate, find threshold, save model and verify
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_recall_curve

# ── Calibration ───────────────────────────────────────────────────────────────
calibrated_model = CalibratedClassifierCV(model, method='isotonic', cv='prefit')
calibrated_model.fit(X_test, y_test)
print('Calibration complete.')

# ── Threshold optimisation on calibrated probabilities ────────────────────────
y_prob_cal    = calibrated_model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_cal)
f1_scores     = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx      = np.argmax(f1_scores[:-1])
CHOSEN_THRESHOLD = float(thresholds[best_idx])

y_pred_final  = (y_prob_cal >= CHOSEN_THRESHOLD).astype(int)
final_f1      = f1_score(y_test, y_pred_final)
final_auc     = roc_auc_score(y_test, y_prob_cal)
print(f'Threshold: {CHOSEN_THRESHOLD:.4f}  →  F1={final_f1:.4f}  AUC={final_auc:.4f}')

# ── Save ──────────────────────────────────────────────────────────────────────
os.makedirs(EVAL_DIR, exist_ok=True)

artifact = {
    'model':            calibrated_model,
    'threshold':        CHOSEN_THRESHOLD,
    'train_features':   TRAIN_FEATURES,
    'dropped_features': BYTECODE_FEATURES,
    'drop_reason': (
        'All phishing contracts in training data are self-destructed. '
        'Bytecode-derived features encode the label, not generalizable signal.'
    ),
    'test_f1':  round(final_f1,  4),
    'test_auc': round(final_auc, 4)
}
joblib.dump(artifact, MODEL_PATH)
print(f'Saved: {MODEL_PATH}')

# ── Verify round-trip ─────────────────────────────────────────────────────────
check      = joblib.load(MODEL_PATH)
y_chk_prob = check['model'].predict_proba(X_test)[:, 1]
y_chk_pred = (y_chk_prob >= check['threshold']).astype(int)
assert np.array_equal(y_chk_pred, y_pred_final), 'Round-trip mismatch'

print(f"Verified — threshold={check['threshold']:.4f}  "
      f"F1={check['test_f1']}  AUC={check['test_auc']}")
print(f"Training features ({len(check['train_features'])}): {check['train_features']}")
print('Notebook 05 complete.')


Calibration complete.
Threshold: 0.5000  →  F1=0.8787  AUC=0.9279
Saved: /content/drive/MyDrive/PhishGuard/models/contract_xgboost_v1.pkl
Verified — threshold=0.5000  F1=0.8787  AUC=0.9279
Training features (9): ['is_verified', 'abi_function_count', 'external_public_function_count', 'approval_related_function_flag', 'permit_related_function_flag', 'setApprovalForAll_flag', 'slither_warning_count_total', 'slither_low_level_call_count', 'slither_access_control_issues_count']
Notebook 05 complete.


In [12]:
# ── Live Contract Prediction ──────────────────────────────────────────────────
import requests, json, re, os, time, joblib
import numpy as np

CONTRACT_ADDRESS  = '0x0000626d6DC72989e3809920C67D01a7fe030000'   # <-- paste contract address here
ETHERSCAN_API_KEY = 'WI9RMPKI74VKDCZ8PHUYF1KK36RV8CVZN2'         # <-- paste your Etherscan key here
RUN_SLITHER       = False                 # True adds ~2 min for verified contracts
MODEL_PATH        = f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl'

# ── Load model artifact ───────────────────────────────────────────────────────
artifact         = joblib.load(MODEL_PATH)
pred_model       = artifact['model']
threshold        = artifact['threshold']
train_features   = artifact['train_features']

# ── Fetch contract data from Etherscan ────────────────────────────────────────
def fetch_contract(address, api_key):
    r = requests.get('https://api.etherscan.io/v2/api', params={
        'chainid': 1, 'module': 'contract',
        'action': 'getsourcecode',
        'address': address, 'apikey': api_key
    }, timeout=15)
    time.sleep(0.25)
    result = r.json().get('result', [{}])[0]
    return result

print(f'Fetching data for {CONTRACT_ADDRESS} ...')
data = fetch_contract(CONTRACT_ADDRESS, ETHERSCAN_API_KEY)

source_code  = data.get('SourceCode', '')
abi_str      = data.get('ABI', '[]')
compiler_ver = data.get('CompilerVersion', '')
is_verified  = int(bool(source_code and source_code not in ('', '0x')))

# ── Extract ABI features ──────────────────────────────────────────────────────
try:
    abi_list = json.loads(abi_str) if abi_str not in ('', 'Contract source code not verified') else []
except Exception:
    abi_list = []

functions  = [item for item in abi_list if item.get('type') == 'function']
func_names = [f.get('name', '').lower() for f in functions]

abi_function_count             = len(functions)
external_public_function_count = sum(
    1 for f in functions if f.get('stateMutability') not in ['view', 'pure'])
approval_related_function_flag = int(any(
    any(kw in n for kw in ['approve','setallowance','increaseallowance','decreaseallowance'])
    for n in func_names))
permit_related_function_flag   = int(any('permit' in n for n in func_names))
setApprovalForAll_flag         = int(any(n == 'setapprovalforall' for n in func_names))

# ── Slither features ──────────────────────────────────────────────────────────
slither_warning_count_total         = 0
slither_low_level_call_count        = 0
slither_access_control_issues_count = 0

if RUN_SLITHER and is_verified and source_code:
    print('Running Slither (this may take up to 2 minutes)...')
    import tempfile, shutil, subprocess

    def install_solc(ver_str):
        match = re.search(r'v?([\d]+\.[\d]+\.[\d]+)', ver_str)
        ver   = match.group(1) if match else '0.8.19'
        os.environ['PATH'] += ':/root/.local/bin:/usr/local/bin'
        os.system(f'solc-select install {ver} 2>/dev/null && solc-select use {ver} 2>/dev/null')

    def run_slither_inline(source_str, address, compiler_ver):
        install_solc(compiler_ver)
        temp_dir = tempfile.mkdtemp()
        src = source_str.strip()
        if src.startswith('{{'):
            try:
                parsed = json.loads(src[1:-1])
                entry  = None
                for fname, content in parsed.get('sources', {}).items():
                    fp = os.path.join(temp_dir, os.path.basename(fname))
                    with open(fp, 'w') as f: f.write(content.get('content', ''))
                    if entry is None: entry = fp
            except Exception:
                shutil.rmtree(temp_dir, ignore_errors=True)
                return 0, 0, 0
        else:
            entry = os.path.join(temp_dir, f'{address}.sol')
            with open(entry, 'w') as f: f.write(src)

        runner = tempfile.mktemp(suffix='.py')
        output = tempfile.mktemp(suffix='.json')
        script = f"""
import json
try:
    from slither.slither import Slither
    sl      = Slither({repr(entry)})
    results = sl.run_detectors()
    total   = sum(len(r.get('elements',[])) for r in results)
    low_lvl = sum(len(r.get('elements',[])) for r in results if 'low-level-calls' in r.get('check',''))
    access  = sum(len(r.get('elements',[])) for r in results if 'access-control' in r.get('check','') or 'unprotected' in r.get('check',''))
    with open({repr(output)},'w') as f: json.dump({{'t':total,'l':low_lvl,'a':access}},f)
except Exception:
    with open({repr(output)},'w') as f: json.dump({{'t':0,'l':0,'a':0}},f)
"""
        with open(runner, 'w') as f: f.write(script)
        try:
            subprocess.run(['python3', runner], timeout=120, capture_output=True)
            with open(output) as f: d = json.load(f)
            return d.get('t',0), d.get('l',0), d.get('a',0)
        except Exception:
            return 0, 0, 0
        finally:
            for fp in [runner, output]:
                try: os.unlink(fp)
                except: pass
            shutil.rmtree(temp_dir, ignore_errors=True)

    slither_warning_count_total, slither_low_level_call_count, \
        slither_access_control_issues_count = run_slither_inline(
            source_code, CONTRACT_ADDRESS.lower(), compiler_ver)
    print(f'Slither done — warnings={slither_warning_count_total}  '
          f'low-level={slither_low_level_call_count}  '
          f'access={slither_access_control_issues_count}')

# ── Build feature vector ──────────────────────────────────────────────────────
feature_map = {
    'is_verified':                           is_verified,
    'abi_function_count':                    abi_function_count,
    'external_public_function_count':        external_public_function_count,
    'approval_related_function_flag':        approval_related_function_flag,
    'permit_related_function_flag':          permit_related_function_flag,
    'setApprovalForAll_flag':                setApprovalForAll_flag,
    'slither_warning_count_total':           slither_warning_count_total,
    'slither_low_level_call_count':          slither_low_level_call_count,
    'slither_access_control_issues_count':   slither_access_control_issues_count
}

X_live = np.array([[feature_map[f] for f in train_features]])

# ── Predict ───────────────────────────────────────────────────────────────────
prob  = pred_model.predict_proba(X_live)[0, 1]
label = int(prob >= threshold)

print()
print('=' * 50)
print(f'Contract : {CONTRACT_ADDRESS}')
print(f'Verified : {"Yes" if is_verified else "No"}')
print(f'ABI fns  : {abi_function_count}  (external: {external_public_function_count})')
print(f'Approval fn flag : {bool(approval_related_function_flag)}')
print('-' * 50)
print(f'Phishing probability : {prob:.4f}')
print(f'Threshold            : {threshold:.4f}')
print(f'Prediction           : {"⚠ PHISHING" if label == 1 else "✓ BENIGN"}')
print('=' * 50)

if not RUN_SLITHER and is_verified:
    print('Note: Slither not run — set RUN_SLITHER=True for full analysis.')


Fetching data for 0x0000626d6DC72989e3809920C67D01a7fe030000 ...

Contract : 0x0000626d6DC72989e3809920C67D01a7fe030000
Verified : No
ABI fns  : 0  (external: 0)
Approval fn flag : False
--------------------------------------------------
Phishing probability : 0.9722
Threshold            : 0.5000
Prediction           : ⚠ PHISHING
